Load the data

In [ ]:
# 
from Functional_Connectivity import compute_fc_directory, save_fc_results

directory = "C:\\Users\\oddafo\\Downloads\\Neutral_movie_1000_cortical\\YA\\ROIsignal"
save_folder = "My_FC_matrices" 

results, run_id = compute_fc_directory(directory)
print(results[0]["fc_matrix"].shape)
print(f"Run ID: {run_id}")

save_folder_file = save_folder + f"\\FC_Y_run_{run_id}.mat"
save_fc_results(results, output_file=save_folder_file, run_id=run_id)


In [ ]:
from scipy.io import loadmat
import re
import numpy as np

path = "My_FC_matrices\\FC_Y_run_1.mat"
data1 = loadmat(path)

filepath = "Input Data\\YoungAge_combined_run2_400_noZ_clean.mat"
data2 = loadmat(filepath)

# --- Extract subject IDs from data1 ---
subject_names1_raw = data1['subject_names'].flatten()
subject_names1 = [int(name[0]) for name in subject_names1_raw]

# --- Extract subject IDs from data2 ---
subject_names2_raw = data2['subject_names'].flatten()
subject_names2 = []
for name in subject_names2_raw:
    subject_id = re.search(r"sub-(.*?)_task", name[0])
    subject_names2.append(int(subject_id.group(1)))

# --- Find missing subjects ---
missing_subjects = set(subject_names1) - set(subject_names2)
print(f"Missing subjects in data2: {missing_subjects}")

# --- Find all indices to remove ---
indices_to_remove = [
    i for i, subj in enumerate(subject_names1) if subj in missing_subjects
]

print(f"Indices to remove from data1: {indices_to_remove}")

# --- Remove them from data1 ---
if indices_to_remove:
    data1['subject_names'] = np.delete(data1['subject_names'][0], indices_to_remove, axis=0)
    data1['fc_matrices'] = np.delete(data1['run1_data'], indices_to_remove, axis=0)

# Save the modified data1 back to the same .mat file
from scipy.io import savemat
savemat(path, data1)

In [ ]:
# Rename the files from "Models\\model_DCEC_1000x1000_72_O_subjects_run1_Cluster_{x}.pt" to "Models\+model_DCEC_1000x1000_68_Y_subjects_run1_Cluster_{x}.pt"

import os

folder_path = "Clusters"

original_pattern = r"DCEC_400x400_72_Y_subjects_run"
new_pattern = r"DCEC_400x400_71_Y_subjects_run"
for filename in os.listdir(folder_path):
    if filename.startswith("DCEC_400x400_72_Y_subjects_run") and filename.endswith("labels_.txt"):
        new_filename = filename.replace(original_pattern, new_pattern)
        os.rename(os.path.join(folder_path, filename), os.path.join(folder_path, new_filename))